In [1]:
import copy
import os, argparse, json, shutil
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1, 2, 3" # for Pytorch DistributedDataParallel(DDP) training
import torch
from torch import optim
from torch.utils.data.distributed import DistributedSampler # for Pytorch DistbutedDataParallel(DDP) training

from colabsfm.roitr.lib.utils import setup_seed
from colabsfm.roitr.configs.utils import load_config
from easydict import EasyDict as edict
from colabsfm.roitr.dataset.dataloader import get_dataset, get_dataloader
from colabsfm.roitr.model.RIGA_v2 import create_model
from colabsfm.roitr.lib.loss import OverallLoss, Evaluator,EvaluatorRegistration
from colabsfm.roitr.lib.tester import get_trainer
from colabsfm.models import SimTr
import numpy as np

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# set up validation
config = load_config("sfmreg/roitr/configs/val/quad.yaml")
config['device'] = torch.device('cuda', 0)
##########################################################
setup_seed(42) # fix the seed

config_sfmreg = edict(config)
# config_sfmreg.pretrain = "workspace/y23w47_sfmreg_combined_lo/y23w47_sfmreg_combined_lo/model_best_PIR.pth"
config_sfmreg.pretrain = 'workspace/y24w7_sfmreg_only/model_best_PIR.pth'
experiment_conf = edict(
    name = 'omniglue',  # just for interfacing
    descriptor_dim = 256,
    lr = 16e-5,
    refiner_num_heads = 1, 
    refiner_n_layers = 4, 
    refiner_descriptor_dim = 64,
    glue_detach = True,
    glue_refine = True,
    flash = True,
    fine_loss_use_mnn = True,
    iter_size = 4,
    low_overlap = False,
    # sfmreg_mode = "sim3",
)
config_sfmreg.update(experiment_conf)

In [3]:
##################################################################
# create model
model = create_model(config_sfmreg).to(config_sfmreg.device)
state = torch.load(config_sfmreg.pretrain)
model.load_state_dict(state['state_dict'])
model = SimTr(model).eval()

In [4]:
##################################################################
# create model roitr
config_roitr = edict(config)
config_roitr.pretrain = "pretrained/model_3dmatch.pth"
######## Experiment settings ###########
experiment_conf = edict(
    lr = 4e-5,
    use_glue = False,
    glue_refine = False,
)
config_roitr.update(experiment_conf)
roitr_model = create_model(config_roitr).to(config_roitr.device)
state = torch.load(config_roitr.pretrain)
roitr_model.load_state_dict({k.replace('module.', ''): v for k, v in state['state_dict'].items()})
roitr_model = SimTr(roitr_model).eval()

In [5]:
from colabsfm.models.pred_wrapper import PreadtorWrap, calibrate_neighbors
import sys
sys.path.append("sfmreg/OverlapPredator")
from models.architectures import KPFCNN
from configs.models import architectures

#############################################
# create Predator model
config = load_config("sfmreg/OverlapPredator/configs/val/sfmreg.yaml")
config['device'] = torch.device('cuda', 0)
config_predator = edict(config)
config_predator.pretrain = "pretrained/predator-3dmatch.pth"
architecture = "indoor"
config_predator.architecture = architectures[architecture]
model_predator = PreadtorWrap(KPFCNN(config_predator).to(config_roitr.device), config_predator)
config_predator.neighborhood_limits = [225, 31, 32, 30]

In [6]:
evaluator = EvaluatorRegistration(config_sfmreg,True)

In [7]:
method = "sift-sattler"#"sp+sg"#"sift-sattler" #"sfm_disk+lightglue" #
scenes = ["GreatCourt", "KingsCollege", "OldHospital","ShopFacade", "StMarysChurch"]
print("Doing cambridge benchmark with SE(3) (shared_scale = True), set shared_scale = False if you want Sim3 benchmark")
for scene in scenes:
    points_A = np.load(f"data/sfmreg/cambridge/{scene}_benchmark/pointclouds/train/{method}_cloud.npy")
    viewpoints_A = np.load(f"data/sfmreg/cambridge/{scene}_benchmark/pointclouds/train/{method}_viewpoints.npy")
    rot_ab = np.eye(3)
    tr_ab = np.zeros([3])
    if True:
        # print("Sampling Random Rotation, Note: For rotation variant methods this needs to be repeated many times to get a reliable estimate")
        from scipy.spatial.transform import Rotation
        euler_ab=np.random.rand(3)*np.pi*2 # anglez, angley, anglex
        rot_ab= Rotation.from_euler('zyx', euler_ab).as_matrix()
        points_A = np.ascontiguousarray((np.matmul(rot_ab,points_A.T).T))
        viewpoints_A = np.ascontiguousarray((np.matmul(rot_ab,viewpoints_A.T).T))
    points_B = np.load(f"data/sfmreg/cambridge/{scene}_benchmark/pointclouds/test/{method}_cloud.npy")
    viewpoints_B = np.load(f"data/sfmreg/cambridge/{scene}_benchmark/pointclouds/test/{method}_viewpoints.npy")
    gt = {"trans": torch.from_numpy(tr_ab.T[None]).float().cuda(), 
        "rot": torch.from_numpy(rot_ab.T[None]).float().cuda()}

    #sfmroitr
    outputs = model.register(torch.from_numpy(points_A).cuda(), torch.from_numpy(points_B).cuda(), 
                                        viewpoints_A = viewpoints_A, viewpoints_B = viewpoints_B, shared_scale = True
                                    )
    evaluator_stats = evaluator(outputs, gt)
    
    # roitr
    outputs_roitr = roitr_model.register(torch.from_numpy(points_A).cuda(), torch.from_numpy(points_B).cuda(), 
                                        viewpoints_A = viewpoints_A, viewpoints_B = viewpoints_B, shared_scale = True
                                    )
    evaluator_stats_roitr = evaluator(outputs_roitr, gt)
    
    # predator
    outputs_predator = model_predator.register(torch.from_numpy(points_A).cuda(), torch.from_numpy(points_B).cuda(), 
                                        viewpoints_A = viewpoints_A, viewpoints_B = viewpoints_B, shared_scale = True
                                    )
    evaluator_stats_predator = evaluator(outputs_predator, gt)
    
    
    #save
    torch.save(gt, 'results_cambridge/model_input'+scene+'.pth')
    torch.save(outputs, 'results_cambridge/model_output'+scene+'.pth')
    torch.save(evaluator_stats, 'results_cambridge/model_stats'+scene+'.pth')
    torch.save(outputs_roitr, 'results_cambridge/roitr_model_output'+scene+'.pth')
    torch.save(evaluator_stats_roitr, 'results_cambridge/roitr_model_stats'+scene+'.pth')
    torch.save(outputs_predator, 'results_cambridge/predator_model_output'+scene+'.pth')
    torch.save(evaluator_stats_predator, 'results_cambridge/predator_model_stats'+scene+'.pth')

Doing cambridge benchmark with SE(3) (shared_scale = True), set shared_scale = False if you want Sim3 benchmark


/home/edanamt/sfmreg/sfmreg/pointops.py:40: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:83.)
  idx = torch.cuda.IntTensor(m, nsample).zero_()
/home/edanamt/sfmreg/sfmreg/roitr/lib/utils.py:562: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3614.)
  src_nodes = torch.matmul(src_nodes, rot.T) + trans.T
/home/edanamt/sfmreg/sfmreg/roitr/lib/loss.py:427: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will 

errors scaled, no transform tensor(0., device='cuda:0')
251 num corrs
RegistrationResult with fitness=1.992032e-02, inlier_rmse=4.492292e-02, and correspondence_set size of 5
Access transformation to get result.
errors scaled, no transform tensor(0., device='cuda:0')
288 num corrs
RegistrationResult with fitness=9.027778e-02, inlier_rmse=3.569494e-02, and correspondence_set size of 26
Access transformation to get result.
errors scaled, no transform tensor(0., device='cuda:0')
305 num corrs
RegistrationResult with fitness=9.836066e-03, inlier_rmse=4.276506e-02, and correspondence_set size of 3
Access transformation to get result.
errors scaled, no transform tensor(0., device='cuda:0')
214 num corrs
RegistrationResult with fitness=5.607477e-02, inlier_rmse=3.564417e-02, and correspondence_set size of 12
Access transformation to get result.
errors scaled, no transform tensor(0., device='cuda:0')
245 num corrs
RegistrationResult with fitness=1.224490e-02, inlier_rmse=3.445849e-02, and corr